# Wasserstein-2 Distance and Optimal Transport

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from monge_ampere.optimal_transport import solve_ot, wasserstein2
from monge_ampere.boundary import BoundaryCondition
import warnings
warnings.filterwarnings("ignore")

## Define Source and Target Densities
Let's transport a Gaussian density to a translated Gaussian density to demonstrate optimal transport. We use the periodic bounded domain $[0,1)^2$.

In [ ]:
n = 64
h = 1.0 / n
x = np.arange(n) * h
X, Y = np.meshgrid(x, x, indexing="ij")

def make_gaussian(X, Y, mu_x, mu_y, sigma, h):
    """Periodic gaussian to avoid boundary cutoff issues."""
    rho = np.zeros_like(X)
    for di in [-1, 0, 1]:
        for dj in [-1, 0, 1]:
            dx = X - mu_x - di
            dy = Y - mu_y - dj
            rho += np.exp(-(dx**2 + dy**2) / (2 * sigma**2))
    rho /= (np.sum(rho) * h * h)
    return rho

sigma = 0.08
d = 4.0 * h
rho0 = make_gaussian(X, Y, 0.5, 0.5, sigma, h)
rho1 = make_gaussian(X, Y, 0.5 + d, 0.5, sigma, h)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].imshow(rho0.T, origin="lower", extent=[0, 1, 0, 1], cmap="Blues")
axes[0].set_title(r"Source Density $\rho_0$")
axes[1].imshow(rho1.T, origin="lower", extent=[0, 1, 0, 1], cmap="Oranges")
axes[1].set_title(r"Target Density $\rho_1$")
plt.show()

## Solve Optimal Transport
We solve the Monge-Ampère equation using the Newton solver inside our package. The solver finds the perturbation potential $\psi$ and the constant mean shift such that det$(D^2\varphi) = \rho_0 / \rho_1(T(x))$.

In [ ]:
result = solve_ot(
    rho0, 
    rho1, 
    h, 
    solver="newton", 
    dw=1, 
    max_iter=30, 
    ot_max_iter=5, 
    damping=0.5
)

w2 = np.sqrt(result.w2_squared)
print(f"Calculated W2 Distance: {w2:.5f}")
print(f"Expected theoretical W2 Distance (from translation): {d:.5f}")

## Visualize the Transport Map and Potential
The transport map $T(x) = \nabla \varphi(x)$ maps $\rho_0$ to $\rho_1$. Because of the periodic domain, we solve for $\psi$, where $\varphi(x) = \frac{1}{2}|x|^2 + \psi(x)$. Therefore $T(x) = x + \nabla\psi(x)$.
We visualize the displacement field $T(x) - x$ below.

In [ ]:
Tx, Ty = result.transport_map
disp_x = Tx - X
disp_y = Ty - Y

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

im0 = axes[0].imshow(result.psi.T, origin="lower", extent=[0, 1, 0, 1], cmap="viridis")
axes[0].set_title(r"Periodic Perturbation Potential $\psi$")
plt.colorbar(im0, ax=axes[0])

# Downsample the vector field for clarity
step = 4
axes[1].quiver(X[::step, ::step], Y[::step, ::step], 
               disp_x[::step, ::step], disp_y[::step, ::step],
               angles='xy', scale_units='xy', scale=1.0, color='r')
axes[1].imshow(rho0.T, origin="lower", extent=[0, 1, 0, 1], cmap="Blues", alpha=0.5)
axes[1].set_title(r"Displacement Field $T(x) - x$ over $\rho_0$")
plt.show()

As we can verify above, the displacement vectors correctly predict the horizontal push towards the target density.